# Day 5 - Three-Asset Direct Autocallable and Fair-Coupon Engine

## tl;dr

**Day 5 gate: PASS.**

- RC-L Table 7 maximum direct-price error versus the paper MC: **0.7614** per 100; maximum fair-coupon error: **1.0419%**.
- Lee Table 8 maximum direct-price error: **0.2994** per 100.
- RC-A value at its frozen **8.7500%** coupon: **94.0231** per 100; `C*` estimate: **14.9151%** (replication SE **0.1582%**, 95% CI **[14.6052%, 15.2251%]**).
- Maximum component identity error: **1.421e-14**; probability-mass error: **0.000e+00**; fair-coupon residual: **0.000e+00**.
- RC-A 252-step monitoring value differs from the 504-step diagnostic by **-0.0053** per 100.

RC-A is a stylised market-informed research contract, not HSBC `40447DKU1` and not an issuer/dealer market price.


## Context & Methods

This experiment log implements the Day 5 baseline (`M0`) for the two frozen
research contracts. `RC-L` is the Lee et al. (2024) reproduction contract;
`RC-A` is a stylised, market-informed continuous-KI contract and is **not** the
HSBC security or an issuer/dealer market price.

The source benchmark is Lee, Ha, Kong, and Lee (2024), *Valuing three-asset
barrier options and autocallable products via exit probabilities of Brownian
bridge*, N.A.J.E.F. 73, 102174, Tables 6-8 and equations (18)-(24).

### Key Assumptions

- Risk-neutral three-asset GBM with constant rate, dividends, volatilities and
  correlation.
- Discrete autocall and coupon events occur only on exact scheduled dates.
- Continuous KI is approximated by a direct fine grid; this is a transparent
  monitoring approximation, not the Day 6 Brownian-bridge estimator.
- The KI event is tracked only until redemption. Contractual barriers and
  initial references remain fixed.
- RC-L pays simple coupon `C * redemption time`; its Table 8 variants use the
  paper's special 27% no-KI maturity coupon.
- RC-A pays quarterly coupons with memory when the worst-of coupon trigger is
  met. Missed coupons accrue until the next satisfied coupon observation.
- Cash flows are linear in annual coupon `C`, so `C*` is computed from the
  discounted coupon annuity and independently checked with a bracketed solver.
- Issuer credit/funding, liquidity and dealer margin remain outside GBM value.

The additive identity is

\[
V=V_{coupon}+V_{early\ principal}+V_{surviving\ notional}+V_{KI\ loss}.
\]


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import platform
import sys
import tempfile

import numpy as np
import openpyxl
import pandas as pd

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "ap_day5_mpl_cache"))
import matplotlib.pyplot as plt


def find_project_root():
    candidates = []
    override = os.environ.get("AP_PROJECT_ROOT")
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "config" / "core_project_config.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate config/core_project_config.json")


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


PROJECT_DIR = find_project_root()
CONFIG_FILE = PROJECT_DIR / "config" / "core_project_config.json"
CONFIG = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
SOURCE_FILE = PROJECT_DIR / CONFIG["market_data"]["relative_path"]
SOURCE_HASH = sha256_file(SOURCE_FILE)
EXPECTED_HASH = CONFIG["market_data"]["sha256"].upper()
assert SOURCE_HASH == EXPECTED_HASH

sys.path.insert(0, str(PROJECT_DIR / "src"))
from autocallable_direct import (
    bracketed_bisection,
    equicorrelation,
    make_psd_correlation,
    parse_research_contract,
    run_replications,
    summarise_replications,
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "day5_direct_autocallable"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_SEED = 20260805
N_PATHS_LEE = 2**12
N_REPLICATIONS_LEE = 3
N_PATHS_RC_A = 2**12
N_REPLICATIONS_RC_A = 4
STEPS_PER_YEAR = 252
BATCH_SIZE = 2048
LEE_PRICE_TOLERANCE = 1.25
LEE_COUPON_TOLERANCE = 0.0200
COMPONENT_TOLERANCE = 1e-10
ROOT_RESIDUAL_TOLERANCE = 1e-10

pd.set_option("display.precision", 8)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "axes.labelsize": 10,
    "axes.edgecolor": "#475569",
    "axes.labelcolor": "#1f2937",
    "text.color": "#1f2937",
    "xtick.color": "#475569",
    "ytick.color": "#475569",
    "grid.color": "#dbe3ec",
    "grid.linewidth": 0.75,
    "grid.alpha": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})
print(f"Project root: {PROJECT_DIR}")
print(f"Workbook SHA-256 verified: {SOURCE_HASH}")
print(f"Lee budget: N={N_PATHS_LEE:,} x R={N_REPLICATIONS_LEE}; RC-A: N={N_PATHS_RC_A:,} x R={N_REPLICATIONS_RC_A}")


## Data

### 1. Load the frozen market inputs and parse RC-L / RC-A

Cached Bloomberg workbook values are used exactly as in Day 2. RC-A starts from
normalised performance 100 and borrows only the frozen rate, dividends,
volatilities and historical correlation. The workbook is dated after the
contract's stylised time origin, so RC-A is market-informed research input, not
a contemporaneous calibration.


In [ ]:
def is_number(value):
    return (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, bool)
        and np.isfinite(value)
    )


def read_market_inputs(path):
    workbook = openpyxl.load_workbook(path, data_only=True, read_only=False)
    setup = workbook["Setup"]
    underlying_snapshot = workbook["Underlying_Snapshot"]
    underlying_history = workbook["Underlying_History"]
    market_history = workbook["Market_History"]
    issuer_initial = workbook["Issuer_Initial_Value"]

    tickers = [underlying_snapshot.cell(row, 2).value for row in (6, 7, 8)]
    spots = np.array([underlying_snapshot.cell(row, 4).value for row in (6, 7, 8)], dtype=float)
    dividend_yields = np.array([underlying_snapshot.cell(row, 5).value for row in (6, 7, 8)], dtype=float) / 100.0
    volatilities = np.array([underlying_snapshot.cell(row, 8).value for row in (6, 7, 8)], dtype=float) / 100.0
    initial_references = np.array([issuer_initial.cell(row, 3).value for row in (38, 39, 40)], dtype=float)

    histories = []
    for date_column, value_column, ticker in zip((1, 4, 7), (2, 5, 8), tickers):
        values = {}
        for row in range(6, underlying_history.max_row + 1):
            date_value = underlying_history.cell(row, date_column).value
            level = underlying_history.cell(row, value_column).value
            if hasattr(date_value, "year") and is_number(level) and level > 0:
                values[pd.Timestamp(date_value)] = float(level)
        histories.append(pd.Series(values, name=ticker).sort_index())
    prices = pd.concat(histories, axis=1, join="inner").dropna()
    log_returns = np.log(prices / prices.shift(1)).dropna()
    correlation = make_psd_correlation(log_returns.corr().to_numpy())

    rate_rows = []
    for row in range(6, market_history.max_row + 1):
        date_value = market_history.cell(row, 10).value
        rate_pct = market_history.cell(row, 11).value
        if hasattr(date_value, "year") and is_number(rate_pct) and rate_pct > 0:
            rate_rows.append((pd.Timestamp(date_value), float(rate_pct) / 100.0))
    rate_series = pd.Series(dict(rate_rows), name="3Y SOFR proxy").sort_index()
    return {
        "tickers": tickers,
        "spots": spots,
        "dividend_yields": dividend_yields,
        "volatilities": volatilities,
        "initial_references": initial_references,
        "prices": prices,
        "correlation": correlation,
        "risk_free_rate": float(rate_series.iloc[-1]),
        "rate_as_of": rate_series.index[-1],
        "workbook_as_of": pd.Timestamp(setup["B8"].value),
    }


market = read_market_inputs(SOURCE_FILE)
assert all(value > 0 for value in market["spots"])
assert all(value > 0 for value in market["volatilities"])
assert all(value > 0 for value in market["initial_references"])
assert np.linalg.eigvalsh(market["correlation"]).min() > -1e-10

rc_l = parse_research_contract(CONFIG, "RC-L")
rc_a = parse_research_contract(CONFIG, "RC-A")

contract_parser_summary = pd.DataFrame([
    {
        "contract_id": contract.contract_id,
        "label": contract.label,
        "principal": contract.principal,
        "maturity_years": contract.maturity_years,
        "observation_count": len(contract.observation_times),
        "coupon_mode": contract.coupon_mode,
        "coupon_memory": contract.coupon_memory,
        "ki_barrier_ratio": contract.ki_barrier_ratio,
        "first_autocall_time": min(
            time for time, enabled in zip(contract.observation_times, contract.autocall_flags) if enabled
        ),
        "claim_boundary": contract.claim_boundary,
    }
    for contract in (rc_l, rc_a)
])
market_inputs = pd.DataFrame({
    "ticker": market["tickers"],
    "spot": market["spots"],
    "dividend_yield": market["dividend_yields"],
    "volatility": market["volatilities"],
    "initial_reference": market["initial_references"],
})
display(contract_parser_summary)
display(market_inputs)
display(pd.DataFrame(market["correlation"], index=market["tickers"], columns=market["tickers"]))
print(f"Workbook as of: {market['workbook_as_of'].date()}; rate proxy: {market['risk_free_rate']:.4%} as of {market['rate_as_of'].date()}")


## Results

### 2. Reproduce Lee Table 7 with the RC-L direct engine

The paper benchmark has eight symmetric GBM scenarios. Its reported direct MC
uses a much finer discretisation but does not disclose random seeds. The frozen
acceptance bands are therefore 1.25 price points per 100 and 200 bp for `C*`;
replication SD/SE and the error versus both paper MC and quasi-exact values are
retained rather than hidden.


In [ ]:
lee_table7_benchmark = pd.DataFrame([
    [0.03, 0.2, 0.4, 101.1633, 101.1305, 100.7617, 0.0528],
    [0.03, 0.2, 0.6, 101.1891, 101.2225, 101.3865, 0.0462],
    [0.03, 0.3, 0.4, 89.4492, 89.8396, 89.4581, 0.1767],
    [0.03, 0.3, 0.6, 91.2325, 91.6220, 91.1359, 0.1623],
    [0.04, 0.2, 0.4, 100.2301, 100.2447, 99.7513, 0.0624],
    [0.04, 0.2, 0.6, 100.3241, 100.3884, 100.4710, 0.0551],
    [0.04, 0.3, 0.4, 89.3601, 89.3411, 89.0015, 0.1832],
    [0.04, 0.3, 0.6, 91.0598, 91.1095, 90.5555, 0.1718],
], columns=["risk_free_rate", "sigma", "rho", "paper_qe_price", "paper_bb_price", "paper_mc_price", "paper_break_even_coupon"])

table7_replication_frames = []
for row in lee_table7_benchmark.itertuples(index=False):
    scenario_id = f"RC-L-r{row.risk_free_rate:.2f}-s{row.sigma:.1f}-rho{row.rho:.1f}"
    table7_replication_frames.append(run_replications(
        scenario_id=scenario_id,
        contract=rc_l,
        risk_free_rate=row.risk_free_rate,
        dividend_yields=np.zeros(3),
        volatilities=np.full(3, row.sigma),
        correlation=equicorrelation(row.rho),
        annual_coupon=rc_l.baseline_annual_coupon,
        n_paths=N_PATHS_LEE,
        steps_per_year=STEPS_PER_YEAR,
        replications=N_REPLICATIONS_LEE,
        base_seed=BASE_SEED,
        batch_size=BATCH_SIZE,
        common_random_numbers=False,
    ))

lee_table7_replications = pd.concat(table7_replication_frames, ignore_index=True)
lee_table7_summary = summarise_replications(lee_table7_replications)
lee_table7_summary[["risk_free_rate", "sigma", "rho"]] = lee_table7_summary["scenario_id"].str.extract(
    r"r([0-9.]+)-s([0-9.]+)-rho([0-9.]+)"
).astype(float)
lee_table7_comparison = lee_table7_summary.merge(
    lee_table7_benchmark,
    on=["risk_free_rate", "sigma", "rho"],
    how="left",
    validate="one_to_one",
)
lee_table7_comparison["price_error_vs_paper_mc"] = lee_table7_comparison["total_value_mean"] - lee_table7_comparison["paper_mc_price"]
lee_table7_comparison["price_error_vs_paper_qe"] = lee_table7_comparison["total_value_mean"] - lee_table7_comparison["paper_qe_price"]
lee_table7_comparison["coupon_error_vs_paper"] = lee_table7_comparison["fair_coupon_mean"] - lee_table7_comparison["paper_break_even_coupon"]
lee_table7_comparison["price_pass"] = lee_table7_comparison["price_error_vs_paper_mc"].abs() <= LEE_PRICE_TOLERANCE
lee_table7_comparison["coupon_pass"] = lee_table7_comparison["coupon_error_vs_paper"].abs() <= LEE_COUPON_TOLERANCE
display(lee_table7_comparison[[
    "risk_free_rate", "sigma", "rho", "paper_mc_price", "total_value_mean", "total_value_se",
    "price_error_vs_paper_mc", "paper_break_even_coupon", "fair_coupon_mean", "fair_coupon_se",
    "coupon_error_vs_paper", "price_pass", "coupon_pass"
]])


### 3. Reproduce Lee Table 8 contract variants

These variants retain the RC-L engine but override the step-down schedule, set
KI to 50%, and pay the paper's fixed 27% maturity coupon on no-call/no-KI paths.
They are validation variants, not additional frozen research contracts.


In [ ]:
lee_table8_benchmark = pd.DataFrame([
    [1, "90-90-90-90-90-90", 90.2910, 90.2966, 90.2064],
    [2, "95-95-90-90-85-85", 89.0452, 88.7913, 89.2180],
    [3, "95-90-85-80-75-70", 90.7650, 90.4242, 90.8118],
], columns=["type", "schedule", "paper_fdm_price", "paper_bb_price", "paper_mc_price"])
table8_triggers = {
    1: (0.90, 0.90, 0.90, 0.90, 0.90, 0.90),
    2: (0.95, 0.95, 0.90, 0.90, 0.85, 0.85),
    3: (0.95, 0.90, 0.85, 0.80, 0.75, 0.70),
}
table8_replication_frames = []
for type_id, triggers in table8_triggers.items():
    variant = rc_l.with_overrides(
        contract_id=f"RC-L-T8-{type_id}",
        autocall_trigger_ratios=triggers,
        ki_barrier_ratio=0.50,
        fixed_maturity_no_ki_coupon_rate=0.27,
    )
    table8_replication_frames.append(run_replications(
        scenario_id=f"RC-L-T8-{type_id}",
        contract=variant,
        risk_free_rate=0.03,
        dividend_yields=np.zeros(3),
        volatilities=np.full(3, 0.30),
        correlation=equicorrelation(0.50),
        annual_coupon=0.10,
        n_paths=N_PATHS_LEE,
        steps_per_year=STEPS_PER_YEAR,
        replications=N_REPLICATIONS_LEE,
        base_seed=BASE_SEED + 1000,
        batch_size=BATCH_SIZE,
        common_random_numbers=False,
    ))

lee_table8_replications = pd.concat(table8_replication_frames, ignore_index=True)
lee_table8_summary = summarise_replications(lee_table8_replications)
lee_table8_summary["type"] = lee_table8_summary["scenario_id"].str.extract(r"([123])$").astype(int)
lee_table8_comparison = lee_table8_summary.merge(lee_table8_benchmark, on="type", how="left", validate="one_to_one")
lee_table8_comparison["price_error_vs_paper_mc"] = lee_table8_comparison["total_value_mean"] - lee_table8_comparison["paper_mc_price"]
lee_table8_comparison["price_error_vs_fdm"] = lee_table8_comparison["total_value_mean"] - lee_table8_comparison["paper_fdm_price"]
lee_table8_comparison["price_pass"] = lee_table8_comparison["price_error_vs_paper_mc"].abs() <= LEE_PRICE_TOLERANCE
display(lee_table8_comparison[["type", "schedule", "paper_fdm_price", "paper_mc_price", "total_value_mean", "total_value_se", "price_error_vs_paper_mc", "price_pass"]])


### 4. Price RC-A and study fair-coupon sensitivities

All scenario revaluations reuse the same seed sequence (CRN). The grid covers
volatility, correlation, continuous-KI barrier and autocall schedule. The
reported `C*` is an economic GBM output for RC-A only; it is not an HSBC market
coupon or quote.


In [ ]:
identity = np.eye(3)
ones = np.ones((3, 3))
corr_low = make_psd_correlation(0.5 * market["correlation"] + 0.5 * identity)
corr_high = make_psd_correlation(0.5 * market["correlation"] + 0.5 * ones)

early_autocall = rc_a.with_overrides(
    contract_id="RC-A-autocall-early",
    autocall_flags=tuple(True for _ in rc_a.observation_times),
    autocall_trigger_ratios=tuple(1.0 for _ in rc_a.observation_times),
)
delayed_flags = tuple(index >= 3 for index in range(len(rc_a.observation_times)))
delayed_autocall = rc_a.with_overrides(
    contract_id="RC-A-autocall-delayed",
    autocall_flags=delayed_flags,
    autocall_trigger_ratios=tuple(1.0 if enabled else None for enabled in delayed_flags),
)

scenario_specs = [
    ("RC-A-baseline", "baseline", "baseline", rc_a, market["volatilities"], market["correlation"]),
    ("RC-A-vol-75", "volatility", "75%", rc_a, 0.75 * market["volatilities"], market["correlation"]),
    ("RC-A-vol-125", "volatility", "125%", rc_a, 1.25 * market["volatilities"], market["correlation"]),
    ("RC-A-corr-low", "correlation", "low", rc_a, market["volatilities"], corr_low),
    ("RC-A-corr-high", "correlation", "high", rc_a, market["volatilities"], corr_high),
    ("RC-A-KI-65", "KI barrier", "65%", rc_a.with_overrides(contract_id="RC-A-KI-65", ki_barrier_ratio=0.65), market["volatilities"], market["correlation"]),
    ("RC-A-KI-85", "KI barrier", "85%", rc_a.with_overrides(contract_id="RC-A-KI-85", ki_barrier_ratio=0.85), market["volatilities"], market["correlation"]),
    ("RC-A-autocall-early", "autocall schedule", "first date", early_autocall, market["volatilities"], market["correlation"]),
    ("RC-A-autocall-delayed", "autocall schedule", "fourth date", delayed_autocall, market["volatilities"], market["correlation"]),
]

rc_a_replication_frames = []
scenario_metadata_rows = []
for scenario_id, dimension, level, contract, volatilities, correlation in scenario_specs:
    frame = run_replications(
        scenario_id=scenario_id,
        contract=contract,
        risk_free_rate=market["risk_free_rate"],
        dividend_yields=market["dividend_yields"],
        volatilities=volatilities,
        correlation=correlation,
        annual_coupon=rc_a.baseline_annual_coupon,
        n_paths=N_PATHS_RC_A,
        steps_per_year=STEPS_PER_YEAR,
        replications=N_REPLICATIONS_RC_A,
        base_seed=BASE_SEED + 2000,
        batch_size=BATCH_SIZE,
        common_random_numbers=True,
    )
    frame["sensitivity_dimension"] = dimension
    frame["sensitivity_level"] = level
    rc_a_replication_frames.append(frame)
    scenario_metadata_rows.append({"scenario_id": scenario_id, "sensitivity_dimension": dimension, "sensitivity_level": level})

rc_a_sensitivity_replications = pd.concat(rc_a_replication_frames, ignore_index=True)
rc_a_sensitivity_summary = summarise_replications(rc_a_sensitivity_replications).merge(
    pd.DataFrame(scenario_metadata_rows), on="scenario_id", how="left", validate="one_to_one"
)
rc_a_sensitivity_summary["fair_coupon_ci95_low"] = rc_a_sensitivity_summary["fair_coupon_mean"] - 1.96 * rc_a_sensitivity_summary["fair_coupon_se"]
rc_a_sensitivity_summary["fair_coupon_ci95_high"] = rc_a_sensitivity_summary["fair_coupon_mean"] + 1.96 * rc_a_sensitivity_summary["fair_coupon_se"]
display(rc_a_sensitivity_summary[[
    "scenario_id", "sensitivity_dimension", "sensitivity_level", "total_value_mean", "total_value_se",
    "fair_coupon_mean", "fair_coupon_ci95_low", "fair_coupon_ci95_high",
    "continuous_ki_survival_probability_mean", "maturity_survival_probability_mean"
]])


### 5. Check monitoring-grid convergence and coupon-root uncertainty

The grid diagnostic uses a smaller independent budget at 126, 252 and 504
steps/year. It does not turn the direct estimator into a continuous-path truth;
Day 6 supplies the Brownian-bridge comparison. Fair-coupon uncertainty is the
dispersion across independent pricing replications and therefore propagates
sampling error through the nonlinear ratio defining `C*`.


In [ ]:
monitoring_frames = []
for steps_per_year in (126, 252, 504):
    monitoring_frames.append(run_replications(
        scenario_id=f"RC-L-monitor-{steps_per_year}",
        contract=rc_l,
        risk_free_rate=0.03,
        dividend_yields=np.zeros(3),
        volatilities=np.full(3, 0.20),
        correlation=equicorrelation(0.40),
        annual_coupon=rc_l.baseline_annual_coupon,
        n_paths=2**11,
        steps_per_year=steps_per_year,
        replications=3,
        base_seed=BASE_SEED + 3000,
        batch_size=1024,
        common_random_numbers=False,
    ))
    monitoring_frames.append(run_replications(
        scenario_id=f"RC-A-monitor-{steps_per_year}",
        contract=rc_a,
        risk_free_rate=market["risk_free_rate"],
        dividend_yields=market["dividend_yields"],
        volatilities=market["volatilities"],
        correlation=market["correlation"],
        annual_coupon=rc_a.baseline_annual_coupon,
        n_paths=2**11,
        steps_per_year=steps_per_year,
        replications=3,
        base_seed=BASE_SEED + 4000,
        batch_size=1024,
        common_random_numbers=False,
    ))

monitoring_convergence_replications = pd.concat(monitoring_frames, ignore_index=True)
monitoring_convergence_summary = summarise_replications(monitoring_convergence_replications)
monitoring_convergence_summary["monitoring_steps_per_year"] = monitoring_convergence_summary["scenario_id"].str.extract(r"([0-9]+)$").astype(int)
monitoring_convergence_summary["contract_layer"] = monitoring_convergence_summary["scenario_id"].str.extract(r"^(RC-[LA])")
reference_504 = monitoring_convergence_summary.loc[
    monitoring_convergence_summary["monitoring_steps_per_year"] == 504,
    ["contract_layer", "total_value_mean", "fair_coupon_mean"],
].rename(columns={"total_value_mean": "value_504", "fair_coupon_mean": "fair_coupon_504"})
monitoring_convergence_summary = monitoring_convergence_summary.merge(reference_504, on="contract_layer", how="left", validate="many_to_one")
monitoring_convergence_summary["value_difference_vs_504"] = monitoring_convergence_summary["total_value_mean"] - monitoring_convergence_summary["value_504"]
monitoring_convergence_summary["fair_coupon_difference_vs_504"] = monitoring_convergence_summary["fair_coupon_mean"] - monitoring_convergence_summary["fair_coupon_504"]
display(monitoring_convergence_summary[[
    "contract_layer", "monitoring_steps_per_year", "total_value_mean", "total_value_se",
    "value_difference_vs_504", "fair_coupon_mean", "fair_coupon_se", "fair_coupon_difference_vs_504"
]])

baseline_replications = rc_a_sensitivity_replications.query("scenario_id == 'RC-A-baseline'").copy()
baseline_root_checks = []
for row in baseline_replications.itertuples(index=False):
    root = bracketed_bisection(lambda coupon: row.base_value + coupon * row.coupon_annuity - rc_a.principal, 0.0, 0.50)
    baseline_root_checks.append({
        "replication": row.replication,
        "linear_fair_coupon": row.fair_coupon,
        "bracketed_fair_coupon": root,
        "root_difference": root - row.fair_coupon,
        "bracketed_residual": row.base_value + root * row.coupon_annuity - rc_a.principal,
    })
fair_coupon_root_checks = pd.DataFrame(baseline_root_checks)
display(fair_coupon_root_checks)


### 6. Evaluate the Day 5 gate and save auditable outputs

Hard gates follow the plan: Lee direct-price agreement, probability mass,
component identity, coupon-root residual and the RC-A claim boundary. Monitoring
convergence is saved as a diagnostic because direct fine-grid monitoring is not
the continuous Brownian-bridge reference.


In [ ]:
all_replications = pd.concat([
    lee_table7_replications,
    lee_table8_replications,
    rc_a_sensitivity_replications,
], ignore_index=True, sort=False)

redemption_rows = []
baseline_scenarios = {
    lee_table7_replications["scenario_id"].iloc[0]: rc_l,
    "RC-A-baseline": rc_a,
}
for scenario_id, contract in baseline_scenarios.items():
    group = all_replications.loc[all_replications["scenario_id"] == scenario_id]
    for observation_index, observation_time in enumerate(contract.observation_times, start=1):
        redemption_rows.append({
            "scenario_id": scenario_id,
            "contract_id": contract.contract_id,
            "observation_index": observation_index,
            "observation_time_years": observation_time,
            "redemption_probability_mean": group[f"redemption_probability_{observation_index}"].mean(),
            "redemption_probability_sd": group[f"redemption_probability_{observation_index}"].std(ddof=1),
            "is_early_redemption": observation_time < contract.maturity_years - 1e-12,
        })
    redemption_rows.append({
        "scenario_id": scenario_id,
        "contract_id": contract.contract_id,
        "observation_index": 0,
        "observation_time_years": contract.maturity_years,
        "redemption_probability_mean": group["maturity_survival_probability"].mean(),
        "redemption_probability_sd": group["maturity_survival_probability"].std(ddof=1),
        "is_early_redemption": False,
    })
redemption_probabilities = pd.DataFrame(redemption_rows)

baseline_component_summary = pd.concat([
    lee_table7_summary.iloc[[0]].assign(baseline_label="RC-L Table 7 baseline"),
    rc_a_sensitivity_summary.query("scenario_id == 'RC-A-baseline'").assign(baseline_label="RC-A market-informed baseline"),
], ignore_index=True, sort=False)

fair_coupon_replications = pd.concat([
    lee_table7_replications.assign(experiment="Lee Table 7"),
    rc_a_sensitivity_replications.assign(experiment="RC-A sensitivity"),
], ignore_index=True, sort=False)[[
    "experiment", "scenario_id", "replication", "base_value", "coupon_annuity",
    "fair_coupon", "fair_coupon_residual", "n_paths", "steps_per_year", "seed"
]]
fair_coupon_summary = pd.concat([
    lee_table7_summary.assign(experiment="Lee Table 7"),
    rc_a_sensitivity_summary.assign(experiment="RC-A sensitivity"),
], ignore_index=True, sort=False)[[
    "experiment", "scenario_id", "replications", "n_paths_per_replication", "steps_per_year",
    "fair_coupon_mean", "fair_coupon_sd", "fair_coupon_se", "fair_coupon_max_abs_residual"
]]
fair_coupon_summary["fair_coupon_ci95_low"] = fair_coupon_summary["fair_coupon_mean"] - 1.96 * fair_coupon_summary["fair_coupon_se"]
fair_coupon_summary["fair_coupon_ci95_high"] = fair_coupon_summary["fair_coupon_mean"] + 1.96 * fair_coupon_summary["fair_coupon_se"]

component_identity_checks = all_replications[[
    "scenario_id", "replication", "total_value", "additive_sum", "component_identity_error",
    "probability_mass_error", "fair_coupon_residual"
]].copy()

sensitivity_dimensions = set(rc_a_sensitivity_summary["sensitivity_dimension"])
gate_rows = [
    {"criterion": "Frozen workbook and generic RC-L/RC-A parser", "observed": f"hash_match={SOURCE_HASH == EXPECTED_HASH}; contracts={','.join(contract_parser_summary['contract_id'])}", "pass": SOURCE_HASH == EXPECTED_HASH and set(contract_parser_summary["contract_id"]) == {"RC-L", "RC-A"}},
    {"criterion": "RC-L Table 7 direct price within preset tolerance", "observed": f"max_abs_error={lee_table7_comparison['price_error_vs_paper_mc'].abs().max():.6f}; tolerance={LEE_PRICE_TOLERANCE:.2f}", "pass": bool(lee_table7_comparison["price_pass"].all())},
    {"criterion": "RC-L Table 7 fair coupon within preset tolerance", "observed": f"max_abs_error={lee_table7_comparison['coupon_error_vs_paper'].abs().max():.6f}; tolerance={LEE_COUPON_TOLERANCE:.4f}", "pass": bool(lee_table7_comparison["coupon_pass"].all())},
    {"criterion": "Lee Table 8 schedule variants within preset tolerance", "observed": f"max_abs_error={lee_table8_comparison['price_error_vs_paper_mc'].abs().max():.6f}; tolerance={LEE_PRICE_TOLERANCE:.2f}", "pass": bool(lee_table8_comparison["price_pass"].all())},
    {"criterion": "Early-redemption mass plus maturity survival equals one", "observed": f"max_abs_error={all_replications['probability_mass_error'].abs().max():.3e}", "pass": bool(all_replications["probability_mass_error"].abs().max() <= COMPONENT_TOLERANCE)},
    {"criterion": "Additive cash-flow components sum to total value", "observed": f"max_abs_error={all_replications['component_identity_error'].abs().max():.3e}", "pass": bool(all_replications["component_identity_error"].abs().max() <= COMPONENT_TOLERANCE)},
    {"criterion": "Fair-coupon linear and bracketed roots have negligible residual", "observed": f"max_linear={all_replications['fair_coupon_residual'].abs().max():.3e}; max_bracketed={fair_coupon_root_checks['bracketed_residual'].abs().max():.3e}", "pass": bool(max(all_replications["fair_coupon_residual"].abs().max(), fair_coupon_root_checks["bracketed_residual"].abs().max()) <= ROOT_RESIDUAL_TOLERANCE)},
    {"criterion": "Required RC-A sensitivities and C* uncertainty are saved", "observed": f"dimensions={sorted(sensitivity_dimensions)}; baseline_R={len(baseline_replications)}", "pass": {"volatility", "correlation", "KI barrier", "autocall schedule"}.issubset(sensitivity_dimensions) and len(baseline_replications) >= 3 and np.isfinite(baseline_replications["fair_coupon"].std(ddof=1))},
    {"criterion": "RC-A claim is market-informed, not HSBC market price", "observed": rc_a.claim_boundary, "pass": "market-informed" in rc_a.claim_boundary and "not HSBC" in rc_a.claim_boundary},
]
gate_summary = pd.DataFrame(gate_rows)
day5_pass = bool(gate_summary["pass"].all())
gate_status = "PASS" if day5_pass else "LIMITED / FAIL"
display(gate_summary)
print(f"Day 5 gate: {gate_status}")

run_manifest = pd.DataFrame([{
    "run_date": pd.Timestamp.now().isoformat(),
    "gate_status": gate_status,
    "project_root": str(PROJECT_DIR),
    "config": str(CONFIG_FILE.relative_to(PROJECT_DIR)),
    "config_schema_version": CONFIG["schema_version"],
    "workbook": str(SOURCE_FILE.relative_to(PROJECT_DIR)),
    "workbook_sha256": SOURCE_HASH,
    "workbook_as_of": market["workbook_as_of"].date().isoformat(),
    "paper_reference": "Lee et al. (2024), doi:10.1016/j.najef.2024.102174, Tables 6-8",
    "direct_monitoring": "fine grid; observation dates inserted exactly",
    "steps_per_year": STEPS_PER_YEAR,
    "lee_n_paths": N_PATHS_LEE,
    "lee_replications": N_REPLICATIONS_LEE,
    "rc_a_n_paths": N_PATHS_RC_A,
    "rc_a_replications": N_REPLICATIONS_RC_A,
    "base_seed": BASE_SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
}])

output_tables = {
    "contract_parser_summary.csv": contract_parser_summary,
    "market_inputs.csv": market_inputs,
    "lee_table7_benchmark.csv": lee_table7_benchmark,
    "lee_table7_replications.csv": lee_table7_replications,
    "lee_table7_comparison.csv": lee_table7_comparison,
    "lee_table8_benchmark.csv": lee_table8_benchmark,
    "lee_table8_replications.csv": lee_table8_replications,
    "lee_table8_comparison.csv": lee_table8_comparison,
    "rc_a_sensitivity_replications.csv": rc_a_sensitivity_replications,
    "rc_a_sensitivity_summary.csv": rc_a_sensitivity_summary,
    "baseline_component_summary.csv": baseline_component_summary,
    "redemption_probabilities.csv": redemption_probabilities,
    "fair_coupon_replications.csv": fair_coupon_replications,
    "fair_coupon_summary.csv": fair_coupon_summary,
    "fair_coupon_root_checks.csv": fair_coupon_root_checks,
    "monitoring_convergence_replications.csv": monitoring_convergence_replications,
    "monitoring_convergence_summary.csv": monitoring_convergence_summary,
    "component_identity_checks.csv": component_identity_checks,
    "gate_summary.csv": gate_summary,
    "run_manifest.csv": run_manifest,
}
for filename, table in output_tables.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.2), constrained_layout=True)
table7_labels = [
    f"{100 * row.risk_free_rate:.0f}/{100 * row.sigma:.0f}/{row.rho:.1f}"
    for row in lee_table7_comparison.itertuples(index=False)
]
axes[0].errorbar(
    np.arange(len(lee_table7_comparison)),
    lee_table7_comparison["price_error_vs_paper_mc"],
    yerr=1.96 * lee_table7_comparison["total_value_se"],
    fmt="o", capsize=3.5, color="#2F6B9A", markerfacecolor="white",
    markeredgewidth=1.6, linewidth=1.5, label="Estimate ± 95% CI",
)
axes[0].axhline(0.0, color="#334155", linewidth=1.0)
axes[0].axhspan(
    -LEE_PRICE_TOLERANCE, LEE_PRICE_TOLERANCE,
    color="#2A9D8F", alpha=0.13, label=f"±{LEE_PRICE_TOLERANCE:.2f} tolerance",
)
axes[0].set_xticks(np.arange(len(table7_labels)))
axes[0].set_xticklabels(table7_labels, fontsize=8.5, rotation=38, ha="right")
axes[0].set_title("RC-L paper-benchmark pricing error", loc="left", pad=12)
axes[0].set_xlabel("Scenario: r% / sigma% / rho")
axes[0].set_ylabel("Direct MC minus paper MC (per 100)")
axes[0].grid(axis="y")
axes[0].legend(loc="lower left", fontsize=8.5)
axes[0].text(
    0.0, 1.015, f"N={N_PATHS_LEE:,} x R={N_REPLICATIONS_LEE}; bars are 95% replication CI",
    transform=axes[0].transAxes, fontsize=8.5, color="#64748b",
)

scenario_order = [
    "RC-A-baseline", "RC-A-vol-75", "RC-A-vol-125", "RC-A-corr-low", "RC-A-corr-high",
    "RC-A-KI-65", "RC-A-KI-85", "RC-A-autocall-early", "RC-A-autocall-delayed",
]
scenario_labels = {
    "RC-A-baseline": "Baseline",
    "RC-A-vol-75": "Volatility x 0.75",
    "RC-A-vol-125": "Volatility x 1.25",
    "RC-A-corr-low": "Lower correlation",
    "RC-A-corr-high": "Higher correlation",
    "RC-A-KI-65": "KI barrier 65%",
    "RC-A-KI-85": "KI barrier 85%",
    "RC-A-autocall-early": "Earlier autocall",
    "RC-A-autocall-delayed": "Later autocall",
}
plot_sensitivity = rc_a_sensitivity_summary.set_index("scenario_id").loc[scenario_order].reset_index()
y_positions = np.arange(len(plot_sensitivity))
bar_colours = ["#2F6B9A" if value == "RC-A-baseline" else "#D97706" for value in plot_sensitivity.scenario_id]
bars = axes[1].barh(
    y_positions,
    100 * plot_sensitivity["fair_coupon_mean"],
    xerr=100 * 1.96 * plot_sensitivity["fair_coupon_se"],
    color=bar_colours, ecolor="#334155", capsize=3, height=0.62,
)
axes[1].set_yticks(y_positions, [scenario_labels[value] for value in plot_sensitivity.scenario_id])
axes[1].invert_yaxis()
axes[1].set_title("RC-A fair coupon by scenario", loc="left", pad=12)
axes[1].set_xlabel("Annual fair coupon (%)")
axes[1].grid(axis="x")
axes[1].bar_label(bars, labels=[f"{100 * value:.1f}%" for value in plot_sensitivity.fair_coupon_mean], padding=4, fontsize=8.5)
axes[1].text(
    0.0, 1.015, f"N={N_PATHS_RC_A:,} x R={N_REPLICATIONS_RC_A}; whiskers are 95% replication CI",
    transform=axes[1].transAxes, fontsize=8.5, color="#64748b",
)
fig.suptitle(
    "Direct-engine validation and fair-coupon sensitivity",
    x=0.01, ha="left", fontsize=14, fontweight="semibold",
)
fig.savefig(OUTPUT_DIR / "day5_validation_and_fair_coupon.png", bbox_inches="tight", facecolor="white")
plt.show()

component_names = ["coupon_value_mean", "early_redemption_principal_mean", "surviving_notional_mean", "terminal_ki_loss_mean"]
component_labels = ["Coupon", "Early-redemption\nprincipal", "Surviving\nnotional", "KI loss"]
baseline_row = rc_a_sensitivity_summary.query("scenario_id == 'RC-A-baseline'").iloc[0]
component_values = np.array([float(baseline_row[name]) for name in component_names])
starts = np.r_[0.0, np.cumsum(component_values)[:-1]]
ends = starts + component_values
bottoms = np.minimum(starts, ends)
heights = np.abs(component_values)

fig, ax = plt.subplots(figsize=(8.2, 4.9), constrained_layout=True)
x = np.arange(len(component_values) + 1)
component_colours = ["#2A9D8F", "#2F6B9A", "#78B7B2", "#D95C5C"]
ax.bar(x[:-1], heights, bottom=bottoms, color=component_colours, width=0.68)
total_value = float(component_values.sum())
ax.bar(x[-1], total_value, color="#334155", width=0.68)
for index in range(len(component_values) - 1):
    ax.plot([x[index] + 0.34, x[index + 1] - 0.34], [ends[index], ends[index]], color="#94a3b8", linewidth=1.0)
for index, value in enumerate(component_values):
    label_y = ends[index] + (2.0 if value >= 0 else -2.0)
    ax.text(index, label_y, f"{value:+.1f}", ha="center", va="bottom" if value >= 0 else "top", fontsize=9, fontweight="semibold")
ax.text(x[-1], total_value + 2.0, f"{total_value:.1f}", ha="center", va="bottom", fontsize=9.5, fontweight="semibold")
ax.axhline(0.0, color="#334155", linewidth=1.0)
ax.set_xticks(x, component_labels + ["Total value"])
ax.set_ylabel("Present value per 100 notional")
ax.set_title("RC-A value bridge at the frozen 8.75% coupon", loc="left", pad=24, fontsize=14)
ax.text(
    0.0, 1.02, f"Component identity closes to numerical precision; fair coupon = {100 * baseline_row.fair_coupon_mean:.2f}%",
    transform=ax.transAxes, fontsize=9, color="#64748b",
)
ax.grid(axis="y")
ax.set_axisbelow(True)
fig.savefig(OUTPUT_DIR / "day5_rca_value_waterfall.png", bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved {len(output_tables)} audit tables and 2 presentation figures to {OUTPUT_DIR}")


## Takeaways

- The reusable parser and direct engine price both frozen research contracts without mixing the HSBC maturity-only market case into continuous-KI experiments.
- All component, probability-mass and root identities pass at numerical precision; Lee Tables 7/8 are reproduced within the pre-set tolerances recorded in `gate_summary.csv`.
- At the frozen **8.7500%** coupon, RC-A is worth **94.0231** per 100 under this GBM setup. Its fair coupon is **14.9151%** with replication SE **0.1582%**. Volatility, correlation, KI-barrier and autocall-schedule results are retained in `rc_a_sensitivity_summary.csv`.
- Fine-grid monitoring remains an approximation. The 126/252/504 diagnostic is evidence of grid sensitivity, while Day 6 must supply the continuous Brownian-bridge comparison.
- The frozen workbook is dated after RC-A's stylised time origin; therefore the result is market-informed research evidence, not a contemporaneous calibration or executable HSBC quote.
